# 2.2 · Data preprocessing: from messy to usable

**Estimated time:** 30–40 minutes · **Prerequisite:** Lesson 2.1 (rows, columns and tensors).

### What you will learn

- Explain why a dataset may not be ready for machine learning, even when it looks like a table.
- Identify **missing values** and choose a simple method for handling them.
- Separate **features** (inputs) from the **target** (the answer we want to predict).
- Turn category labels into numbers using **one-hot encoding**, then convert the result into a PyTorch tensor.

We continue with our fictional study dataset. This time, imagine Ben forgot to record the number of practice questions. We have also recorded *how* each learner studied (Video or Text), which is a word rather than a number.


## 1. Why cleaning comes before modeling

A computer needs a consistent representation: values must be in the right columns, missing information must have an explicit treatment, and categorical labels must be encoded appropriately. The picture below shows the transformation we will complete in this notebook.


### What are we trying to achieve?

**Data preprocessing** means turning raw observations into a trustworthy, consistent form the model can use. We do it because raw data may contain unknown values, inconsistent units, text labels, or columns that should *not* be used to predict the answer.

**How it works:** (1) inspect the table, (2) decide how missing values will be handled, (3) separate inputs from the answer, (4) encode categories, and (5) optionally scale numeric features. **ML connection:** a model can learn misleading patterns if preprocessing accidentally uses information from the future or test set.


<picture>
  <source type="image/svg+xml" srcset="images/05-cleaning-pipeline.svg">
  <img src="images/05-cleaning-pipeline.png" alt="Three stages: a raw table, cleaned numeric features, and a numerical tensor" loading="lazy">
</picture>

*Figure: Three stages: a raw table, cleaned numeric features, and a numerical tensor.*


Our starting records are:

| Learner | Hours | Practice | Method | Score / 10 |
|:--|--:|--:|:--|--:|
| Alice | 1 | 2 | Video | 3 |
| Ben | 2 | **missing** | Text | 5 |
| Chika | 3 | 6 | Video | 7 |
| Dara | 4 | 8 | Text | 9 |

A missing value is **not** the same as zero: zero would mean Ben completed *no* practice questions; “missing” means we **do not know**. We must not silently change one into the other.


In [1]:
import pandas as pd
import torch
from IPython.display import display

raw = pd.DataFrame({
    'Learner': ['Alice', 'Ben', 'Chika', 'Dara'],
    'Hours': [1., 2., 3., 4.],
    'Practice': [2., None, 6., 8.],
    'Method': ['Video', 'Text', 'Video', 'Text'],
    'Score': [3., 5., 7., 9.],
})
display(raw)
print('Missing values per column:\n', raw.isna().sum())


,Learner,Hours,Practice,Method,Score
0,Alice,1.0,2.0,Video,3.0
1,Ben,2.0,NaN,Text,5.0
2,Chika,3.0,6.0,Video,7.0
3,Dara,4.0,8.0,Text,9.0


Missing values per column:
 Learner     0
Hours       0
Practice    1
Method      0
Score       0
dtype: int64


**What did we observe?** `pandas` displays the missing practice value as `NaN` (“Not a Number”). `raw.isna().sum()` counts missing cells in each column. Here, exactly one Practice value is missing.

## 2. Handling a missing value

There is no universally correct replacement. Common choices are:

- **Drop a row** if it cannot be used, though this loses data and can introduce bias.
- **Fill with a meaningful known value** if you really have one.
- **Fill using an observed statistic** such as a median or mean (a simple baseline, not a magic fix).

For demonstration, we fill Ben's practice count using the **mean of the three observed values**:

$$
\frac{2+6+8}{3} = \frac{16}{3} \approx 5.33
$$

The result is an *estimate*. It is **not** a claim that Ben actually completed 5.33 questions. In a real project, we should record which values were originally missing and evaluate the choice.


### Why choose the mean here, and what could go wrong?

For known counts `2, 6, 8`, the mean is `(2 + 6 + 8) / 3 = 5.33`. Replacing a missing cell with this value is called **mean imputation**. It lets a numerical algorithm accept a complete column, but it introduces an estimated value and may hide differences among learners.

- **Unknown ≠ zero:** zero asserts no practice occurred; missing means the count was unreported.
- **Imputation ≠ recovery:** we have *not* discovered Ben's actual count.
- **Mean ≠ always best:** a median may be more robust to extreme values, and sometimes keeping a separate "was missing" flag is useful.

In real evaluation, compute the fill value on the training portion only. If a test example helps determine the fill value, information from evaluation has leaked into training.


<picture>
  <source type="image/svg+xml" srcset="images/06-missing-values.svg">
  <img src="images/06-missing-values.png" alt="The same four-row table before and after the missing practice value is filled with the observed mean" loading="lazy">
</picture>

*Figure: The same four-row table before and after the missing practice value is filled with the observed mean.*


In [2]:
cleaned = raw.copy()
observed_mean = cleaned['Practice'].mean()  # NaN is ignored by pandas.mean()
cleaned['Practice'] = cleaned['Practice'].fillna(observed_mean)
print('Observed mean:', round(observed_mean, 2))
display(cleaned)
assert cleaned['Practice'].isna().sum() == 0


Observed mean: 5.33


,Learner,Hours,Practice,Method,Score
0,Alice,1.0,2.000000,Video,3.0
1,Ben,2.0,5.333333,Text,5.0
2,Chika,3.0,6.000000,Video,7.0
3,Dara,4.0,8.000000,Text,9.0


**Important for later:** When splitting real data into training and test sets, compute your fill value using **training data only**, then apply that same value to validation/test data. Otherwise information from the test set leaks into model preparation. We'll formalize this in Chapter 3.

## 3. Decide what the model sees and what it should predict

**Features** are pieces of information given to the model, such as Hours, Practice and Method. The **target** is the desired answer: Score.

We must not accidentally include Score among the model's features while asking it to predict Score that would give away the answer. We also exclude the learner's name: it is an identifier, not a numeric explanation of performance.


<picture>
  <source type="image/svg+xml" srcset="images/07-feature-target.svg">
  <img src="images/07-feature-target.png" alt="Inputs enter the model; predictions are compared with separately observed target scores, with each table fully labeled" loading="lazy">
</picture>

*Figure: Inputs enter the model; predictions are compared with separately observed target scores, with each table fully labeled.*


**In the illustration**, the predicted scores `[4, 5, 6, 8]` are hypothetical model outputs. They are not known targets. The known scores `[3, 5, 7, 9]` are the targets we compare those predictions against to measure error.


In [3]:
features = cleaned[['Hours', 'Practice', 'Method']].copy()
target = cleaned['Score'].copy()
print('Features:')
display(features)
print('Target values:', target.to_list())


Features:


,Hours,Practice,Method
0,1.0,2.000000,Video
1,2.0,5.333333,Text
2,3.0,6.000000,Video
3,4.0,8.000000,Text


Target values: [3.0, 5.0, 7.0, 9.0]


### Features versus targets: what can the model see?

A **feature** is a measurement available to the model as an input. A **target** is the answer we ask it to predict. In our toy task, study hours, practice count, and study method are input features; Score is the target.

Why keep the target out of the inputs? If we hand a model the correct score while asking it to predict that same score, its excellent apparent performance would be meaningless. This is an obvious case of **target leakage**. Other leakage can be subtler, such as using information collected *after* the outcome.


## 4. Turning categories into numbers

If Method can be `Video` or `Text`, why not replace those with 1 and 2? Because such numbers may wrongly suggest that “Text is twice Video” or that the categories have an order.

**One-hot encoding** makes a separate column for each category. `Video` becomes `[1, 0]`; `Text` becomes `[0, 1]`. The resulting values only say whether a category applies.

For a beginner exercise we keep **both** indicator columns so the transformation is visible. Some models drop one redundant indicator column; that detail can wait until later.


### Read each encoded row, not just the column names

With category columns `Method_Text` and `Method_Video`, a learner marked "Text" has `(1, 0)` and a learner marked "Video" has `(0, 1)` (the order depends on the actual column names). A **1 means the category applies** and **0 means it does not**. We do not interpret the two columns as quantities that should be added or ordered.

**Why this matters:** many models work with numbers but should not be told that Video is "greater than" Text just because we assigned arbitrary numeric labels. Other encoding schemes exist; one-hot is a clear starting point.


<picture>
  <source type="image/svg+xml" srcset="images/08-one-hot-encoding.svg">
  <img src="images/08-one-hot-encoding.png" alt="A fully labeled four-row table showing categorical labels converted to two binary indicator columns" loading="lazy">
</picture>

*Figure: A fully labeled four-row table showing categorical labels converted to two binary indicator columns.*


In [4]:
numeric_features = pd.get_dummies(features, columns=['Method'], dtype=int)
display(numeric_features)
print('Column names:', numeric_features.columns.to_list())
assert numeric_features.select_dtypes(include='object').empty


,Hours,Practice,Method_Text,Method_Video
0,1.0,2.000000,0,1
1,2.0,5.333333,1,0
2,3.0,6.000000,0,1
3,4.0,8.000000,1,0


Column names: ['Hours', 'Practice', 'Method_Text', 'Method_Video']


## 5. Ready for PyTorch

We can now convert the numeric feature table to a **tensor**. The model would see four rows (learners) and four columns (Hours, Practice, Method_Text, Method_Video): shape **(4, 4)**. The target contains one score per learner: shape **(4,)**.

The `float32` type is widely used for deep-learning calculations. The code below keeps the **inputs and targets separate**.


In [5]:
X = torch.tensor(numeric_features.to_numpy(dtype='float32'))
y = torch.tensor(target.to_numpy(dtype='float32'))

print('Feature columns:', list(numeric_features.columns))
print('X shape:', tuple(X.shape))
print('y shape:', tuple(y.shape))
print('First learner’s features:', X[0].tolist())
print('First learner’s score:', y[0].item())
assert tuple(X.shape) == (4, 4) and tuple(y.shape) == (4,)


Feature columns: ['Hours', 'Practice', 'Method_Text', 'Method_Video']
X shape: (4, 4)
y shape: (4,)
First learner’s features: [1.0, 2.0, 0.0, 1.0]
First learner’s score: 3.0


## 6. A first look at scaling

Some numerical features may have very different magnitudes: study hours might range from 1 to 4, while practice questions might range from 0 to 1,000. In many models, large numerical scales can make optimization harder.

One simple technique is **min–max scaling**: subtract the minimum, then divide by the range. For `[1, 2, 3, 4]`, this produces `[0, 0.333…, 0.667…, 1]`. Scaling changes how a number is represented, not which learner it belongs to.

In real training pipelines, just as with filling missing data, learn scaling parameters from **training data only**.


In [6]:
study_hours = torch.tensor([1., 2., 3., 4.])
scaled_hours = (study_hours - study_hours.min()) / (study_hours.max() - study_hours.min())
print('Before:', study_hours.tolist())
print('After :', [round(v, 3) for v in scaled_hours.tolist()])


Before: [1.0, 2.0, 3.0, 4.0]
After : [0.0, 0.333, 0.667, 1.0]


### Scaling worked example

**What:** Scaling changes the numerical range of a feature while preserving which rows the values belong to. **Why:** wildly different magnitudes can complicate some optimization procedures. **How:** for min–max scaling, use `(value − minimum) / (maximum − minimum)` when the range is nonzero. For Hours `[1,2,3,4]`, the value 3 becomes `(3−1)/(4−1) = 2/3 ≈ 0.667`.

**A key boundary:** if maximum equals minimum, the denominator is zero, so we must handle that case separately. Also, fitted minimum and maximum must come from **training data**, not from held-out evaluation examples. Scaling is a representation choice, not a guarantee of better predictions.


## 7. Practice: explain every column

Before running the next cell, answer:

1. Which part of our table is the **target**, and why should it not be part of `X`?
2. If `Practice` is missing, is it correct to replace it with `0` automatically? Explain.
3. How many feature columns remain if Method has two categories and we keep both indicator columns?
4. Why is filling missing values using statistics from the **test data** a problem?

**Takeaway:** Preprocessing is the step where raw observations become consistent, model-readable **features and targets**. Decisions here affect everything the model learns.

➡️ **Next:** [2.3 Linear algebra](03-linear-algebra.ipynb) — how models combine those numeric features.


In [7]:
# Try changing the category encoding: What if a new method "Audio" appeared?
toy = pd.DataFrame({'Method': ['Video', 'Text', 'Audio', 'Video']})
display(pd.get_dummies(toy, dtype=int))
print('Three categories produce three columns with this full one-hot encoding.')


,Method_Audio,Method_Text,Method_Video
0,0,0,1
1,0,1,0
2,1,0,0
3,0,0,1


Three categories produce three columns with this full one-hot encoding.
